# Featherless Inference Server — running on AMD Developer Cloud

This notebook is what actually dispatches the Featherless chat-completion calls for the
specialist/synthesis agents. It runs on this AMD Jupyter instance and exposes a tiny local
HTTP server (`POST /v1/chat/completions`) that `backend/agent_core.py` calls instead of
hitting `api.featherless.ai` directly from wherever FastAPI happens to be hosted.

Run cells top to bottom, then leave this notebook open/running while you use the backend.

**Run the device-info cell first and keep its output in the committed notebook** — same
convention as `specialist_eval_and_embeddings.ipynb`.

In [ ]:
# --- Device info: confirm this notebook process is actually on the AMD box ---
import subprocess

try:
    print(subprocess.run(["rocm-smi"], capture_output=True, text=True).stdout)
except FileNotFoundError:
    print("rocm-smi not found on PATH (fine if you're smoke-testing this notebook off-AMD).")

try:
    import torch
    print("torch:", torch.__version__)
    print("cuda/rocm available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("device:", torch.cuda.get_device_name(0))
except ImportError:
    print("torch not installed in this kernel yet — pip install -r requirements.txt")

In [ ]:
# --- Install/ensure deps for the local server (safe to re-run) ---
%pip install -q fastapi uvicorn nest_asyncio requests python-dotenv

In [ ]:
# --- Load the same FEATHERLESS_API_KEY the backend uses ---
# Put backend/.env (or a copy of it) next to this notebook, or export the var
# in the AMD Jupyter environment directly. Never hardcode the key here.
import os
from dotenv import load_dotenv

load_dotenv()  # looks for a .env in the notebook's working directory
load_dotenv("../backend/.env")  # also try the backend's .env if this notebook sits in amd_compute/

FEATHERLESS_API_KEY = os.environ.get("FEATHERLESS_API_KEY", "")
FEATHERLESS_MODEL = os.environ.get("FEATHERLESS_MODEL", "Qwen/Qwen2.5-7B-Instruct")
FEATHERLESS_URL = "https://api.featherless.ai/v1/chat/completions"

assert FEATHERLESS_API_KEY, "FEATHERLESS_API_KEY not found — set it in a .env this notebook can see."
print(f"Loaded Featherless config. Model: {FEATHERLESS_MODEL}")

In [ ]:
# --- The actual call, executed from THIS notebook process on the AMD box ---
import time
import requests

def call_featherless(messages: list, temperature: float = 0.2, max_tokens: int = 1000) -> dict:
    """Same retry-on-429/transient-error behavior as backend/agent_core.py's
    _post_with_retry, kept in sync manually since this now runs in a separate
    process/environment from the backend."""
    last_exc = None
    for attempt in range(4):
        try:
            resp = requests.post(
                FEATHERLESS_URL,
                headers={
                    "Authorization": f"Bearer {FEATHERLESS_API_KEY}",
                    "Content-Type": "application/json",
                },
                json={
                    "model": FEATHERLESS_MODEL,
                    "max_tokens": max_tokens,
                    "temperature": temperature,
                    "messages": messages,
                },
                timeout=30,
            )
        except requests.exceptions.RequestException as e:
            last_exc = e
            time.sleep(0.6 * (attempt + 1))
            continue
        if resp.status_code == 429:
            last_exc = RuntimeError(f"Rate limited (429), attempt {attempt + 1}/4")
            time.sleep(1.5 * (attempt + 1))
            continue
        resp.raise_for_status()
        return resp.json()
    raise last_exc if last_exc else RuntimeError("Request failed after retries.")

print("call_featherless() ready.")

In [ ]:
# --- Local HTTP server: mirrors Featherless's own /v1/chat/completions shape ---
# backend/agent_core.py posts here instead of api.featherless.ai directly, so the
# outbound call to Featherless is dispatched from this AMD notebook process.
import nest_asyncio
import uvicorn
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse

nest_asyncio.apply()

server_app = FastAPI(title="Featherless-on-AMD relay")

@server_app.post("/v1/chat/completions")
async def chat_completions(request: Request):
    body = await request.json()
    messages = body.get("messages", [])
    temperature = body.get("temperature", 0.2)
    max_tokens = body.get("max_tokens", 1000)
    try:
        result = call_featherless(messages, temperature=temperature, max_tokens=max_tokens)
        return JSONResponse(result)
    except Exception as e:
        return JSONResponse({"error": str(e)}, status_code=502)

@server_app.get("/health")
async def health():
    return {"status": "ok", "model": FEATHERLESS_MODEL}

print("Server app defined. Run the next cell to start it (blocks the kernel while running).")

In [ ]:
# --- Start the server. Leave this cell running while backend/agent_core.py calls it. ---
# Bind 0.0.0.0 if the FastAPI backend runs on a different machine/container than this
# notebook and needs to reach it over the network; otherwise 127.0.0.1 is fine.
HOST = "0.0.0.0"
PORT = 8800
print(f"Starting Featherless-on-AMD relay on http://{HOST}:{PORT} ...")
uvicorn.run(server_app, host=HOST, port=PORT, log_level="info")